# BMRC: Reproducible evaluation walkthrough

This notebook guides you through preparing local inputs, building the frozen-feature cache, running the main leave-one-generator-out evaluation, and inspecting its outputs. BMRC trains branchwise MSE residual heads on labeled real/generated videos; it is supervised.

**Data note.** The repository does not distribute benchmark videos, DINOv3 weights, STALL reference statistics, or the original private cache. Exact reported results require the same clip IDs, labels, frame indices, checkpoint, and reference archive. With your own licensed files, this notebook runs a new evaluation and should be reported as such. Keep videos, weights, and caches local.

Run the setup cells once, then execute the experiment cells in order. The training/evaluation cells can take a long time, depending on the number of clips and hardware.

## 1. Environment and required files

Create the conda environment from the repository root and install dependencies. Install a PyTorch build matching your CUDA driver from [the official PyTorch selector](https://pytorch.org/get-started/locally/). Obtain the DINOv3 repository/checkpoint and STALL reference archive under their respective terms.

In [ ]:
# Run from the repository root in a terminal (not in this notebook):
# conda env create -f environment.yml
# conda activate bmrc-video-detection
# python -m pip install -r requirements.txt
# export DINO_V3_REPO_DIR=/path/to/dinov3
# export DINO_V3_WEIGHTS=/path/to/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
# export STALL_PARAMS=/path/to/stall_params_vatex_dino_v3.npz

## 2. Check GPU and paths

Set the paths below to your local DINOv3 checkout, checkpoint, and real-reference archive. This cell checks that they exist and reports whether CUDA is available.

In [ ]:
import os
from pathlib import Path
import torch

# Jupyter often starts in notebooks/. Resolve the repository root either way.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Open this notebook from inside the cloned BMRC repository.")
os.chdir(ROOT)

DINO_REPO = Path(os.environ.get('DINO_V3_REPO_DIR', '/path/to/dinov3'))
DINO_WEIGHTS = Path(os.environ.get('DINO_V3_WEIGHTS', '/path/to/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'))
STALL_PARAMS = Path(os.environ.get('STALL_PARAMS', '/path/to/stall_params_vatex_dino_v3.npz'))

print('Repository root:', ROOT)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
for label, path in [('DINOv3 repo', DINO_REPO), ('DINOv3 checkpoint', DINO_WEIGHTS), ('STALL reference archive', STALL_PARAMS)]:
    print(f'{label}: {path} —', 'found' if path.exists() else 'MISSING; update the path before running cache preparation')

## 3. Prepare a video manifest

Copy `manifests/manifest_template.csv` and fill one row per clip. Use `subset=real` for authentic videos and a generator name for each generated source. `frame_indices_json` is a JSON list of zero-based frame indices; the primary setting uses 16 frames sampled at 8 FPS from two-second clips. Every `video_path` must point to a local file. Exact replay requires the original rows and selected frame indices.

In [ ]:
from pathlib import Path
import pandas as pd

manifest = ROOT / 'manifests/my_manifest.csv'
if manifest.exists():
    df = pd.read_csv(manifest)
    display(df.head())
    required = {'id','benchmark','source_model','subset','duration','video_path','frame_indices_json'}
    print('Missing columns:', sorted(required - set(df.columns)))
    print('Rows:', len(df), '| IDs unique:', df['id'].is_unique)
else:
    print('Create', manifest, 'by copying manifests/manifest_template.csv and adding your licensed clips.')

## 4. Build the feature cache

This extracts frozen frame embeddings and spatial/temporal likelihoods. It writes generated files under `data/`; `.gitignore` prevents caches and video data from being committed.

In [ ]:
import os
import subprocess
import sys

manifest = ROOT / 'manifests/my_manifest.csv'
cache_dir = ROOT / 'data/my_run'
RUN_CACHE = False  # Set True after the manifest, DINOv3 files, and STALL_PARAMS are ready.
if RUN_CACHE:
    if not manifest.is_file():
        raise FileNotFoundError(f"Create the local manifest first: {manifest}")
    if not STALL_PARAMS.is_file() or not DINO_REPO.is_dir() or not DINO_WEIGHTS.is_file():
        raise FileNotFoundError("Set DINO_V3_REPO_DIR, DINO_V3_WEIGHTS, and STALL_PARAMS to valid local paths.")
    command = [sys.executable, str(ROOT / 'src/prepare_cache.py'), str(manifest),
               '--output', str(cache_dir), '--params', str(STALL_PARAMS),
               '--dino-repo', str(DINO_REPO), '--dino-weights', str(DINO_WEIGHTS),
               '--device', 'cuda' if torch.cuda.is_available() else 'cpu']
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Cache extraction is disabled. Set RUN_CACHE=True after checking the local inputs above.')

## 5. Run the experiments

The primary script trains on other generator groups within each benchmark and evaluates on the held-out group. It compares BMRC and the fixed reference score on the same balanced test clips. The component script evaluates branch and fusion ablations on those folds. Set `BMRC_REPLAY` to the cache produced above.

In [ ]:
import os
import subprocess
import sys

replay = ROOT / 'data/my_run'
main_output = ROOT / 'results/my_run_main'
abl_output = ROOT / 'results/my_run_ablation'
RUN_EXPERIMENTS = False  # Set True after cache extraction has completed.
if RUN_EXPERIMENTS:
    if not (replay / 'records').is_dir() or not (replay / 'features').is_dir():
        raise FileNotFoundError(f"Feature cache is incomplete: {replay}")
    env = os.environ.copy()
    env['BMRC_REPLAY'] = str(replay)
    env['STALL_PARAMS'] = str(STALL_PARAMS)
    env['BMRC_OUTPUT'] = str(main_output)
    subprocess.run([sys.executable, str(ROOT / 'src/run_bmrc_leave_one_generator.py')],
                   cwd=ROOT, env=env, check=True)
    env['BMRC_ABLATION_OUTPUT'] = str(abl_output)
    subprocess.run([sys.executable, str(ROOT / 'src/run_bmrc_component_ablation.py')],
                   cwd=ROOT, env=env, check=True)
else:
    print('Training/evaluation is disabled. Set RUN_EXPERIMENTS=True after cache extraction is complete.')

## 6. Inspect the saved results

The scripts write per-seed and per-generator CSV files. The code reports AUROC and AP with real videos as the positive class. A run on a new manifest is a new experiment; do not describe its metrics as reproducing the saved paper tables unless the required IDs, labels, frame indices, model weights, and reference archive match.

In [ ]:
from pathlib import Path
import pandas as pd

summary = ROOT / 'results/my_run_main/per_generator_summary.csv'
if summary.exists():
    display(pd.read_csv(summary))
else:
    print('Results not found yet. Run the main experiment cell first.')

## Reproducibility and attribution

Record the manifest version, checkpoint and reference-archive checksums, software versions, GPU, and command line alongside every run. Do not upload benchmark videos, derived feature caches, or model weights unless their terms explicitly permit redistribution. This implementation adapts released STALL code and real-reference likelihood components; follow the attribution and license in `licenses/STALL-CC-BY-NC-4.0.txt` and cite the STALL paper. See the repository README for details.